# ⚠️ DEPRECATED — no longer part of the pipeline

This notebook's job (keyword-based classification) was merged into **Notebook 2** along with field extraction, and both now run via the Claude API instead of keyword/regex rules — this fixed real accuracy problems the keyword approach had (documents using different label wording than expected, e.g. "FOR" instead of "Patient Name", were never classified correctly).

This notebook is **not run by the Databricks Job anymore** (removed from the task graph). Kept in the repo for reference/history only — do not re-add it to the job without updating it to match Notebook 2's current table schemas.

In [ ]:
from pyspark.sql import functions as F

catalog = "cdac-project"
schema = "intelligent-main-folder"
extracted_table = f"`{catalog}`.`{schema}`.extracted_text"
classification_table = f"`{catalog}`.`{schema}`.document_classification"

In [ ]:
# Only classify files that were actually extracted successfully —
# nothing to classify for FAILED extractions or empty raw_text.
extracted_df = spark.table(extracted_table).filter(
    (F.col("extraction_status") == "SUCCESS") &
    F.col("raw_text").isNotNull()
)

display(extracted_df)

In [ ]:
# Incremental like the earlier notebooks — skip files that already
# have a classification row so re-running this notebook doesn't
# reclassify (and duplicate-append) everything every time.
if spark.catalog.tableExists(classification_table):
    already_classified_ids = {
        row.file_id
        for row in spark.table(classification_table).select("file_id").collect()
    }
else:
    already_classified_ids = set()

to_classify_df = (
    extracted_df.filter(~F.col("file_id").isin(already_classified_ids))
    if already_classified_ids
    else extracted_df
)

display(to_classify_df)

In [ ]:
# Keyword-based rules, one entry per document type. To support a new
# document type later, just add an entry here — classify_document()
# and everything downstream stays unchanged.
DOCUMENT_TYPE_RULES = {
    "RESUME": [
        "resume", "experience", "education", "skills",
        "certifications", "projects", "achievements",
    ],
    "INVOICE": [
        "invoice", "invoice number", "bill to", "ship to", "amount due",
        "total due", "subtotal", "purchase order", "invoice date", "tax",
    ],
    "BANK_STATEMENT": [
        "account statement", "account number", "opening balance",
        "closing balance", "transaction date", "statement period",
        "ifsc", "account summary", "withdrawal", "deposit",
    ],
    "PRESCRIPTION": [
        "prescription", "rx", "dosage", "physician", "patient name",
        "medicine", "tablet", "mg", "refill", "diagnosis",
    ],
    "EMAIL": [
        "from:", "to:", "subject:", "cc:", "sent:", "dear",
        "regards", "sincerely", "forwarded message", "original message",
        "reply-to:", "wrote:",
    ],
}


def classify_document(raw_text):
    """
    Generic keyword-based document classifier.

    Scores the text against every registered document type's keyword
    list (fraction of that type's keywords found in the text) and
    returns the best match. Returns ("UNKNOWN", 0.0) when nothing
    matches at all.
    """

    if not raw_text:
        return "UNKNOWN", 0.0

    text = raw_text.lower()

    scores = {
        document_type: sum(1 for keyword in keywords if keyword in text) / len(keywords)
        for document_type, keywords in DOCUMENT_TYPE_RULES.items()
    }

    best_type, best_score = max(scores.items(), key=lambda item: item[1])

    if best_score == 0:
        return "UNKNOWN", 0.0

    return best_type, round(best_score, 4)

In [ ]:
results = []

for row in to_classify_df.collect():

    file_id = row["file_id"]
    file_name = row["file_name"]
    raw_text = row["raw_text"]

    document_type, classification_confidence = classify_document(raw_text)

    status = "CLASSIFIED" if document_type != "UNKNOWN" else "UNCLASSIFIED"

    results.append(
        (
            file_id,
            file_name,
            raw_text,
            document_type,
            classification_confidence,
            status
        )
    )

In [ ]:
classification_schema = """
file_id STRING,
file_name STRING,
raw_text STRING,
document_type STRING,
classification_confidence DOUBLE,
status STRING
"""

classification_df = (
    spark.createDataFrame(results, schema=classification_schema)
    .withColumn("classification_timestamp", F.current_timestamp())
)

display(classification_df)

In [ ]:
classification_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(classification_table)

In [ ]:
display(spark.table(classification_table))